# M8–M9 locked test and inference
Run locked walk-forward and date-level statistical inference only after the PTCST protocol lock and main ablations are complete.

In [ ]:
# Bootstrap this notebook even when it is opened in a fresh Colab kernel.
from pathlib import Path
import urllib.request
_BOOTSTRAP_REPO = Path('/content/kltn')
_BOOTSTRAP_SCRIPT = _BOOTSTRAP_REPO / 'scripts' / 'colab_bootstrap.py'
if not _BOOTSTRAP_SCRIPT.exists():
    raw = 'https://raw.githubusercontent.com/maiphuowng205/kltn/fcb0507351d694c2431e454bd84a357958f96635/scripts/colab_bootstrap.py'
    urllib.request.urlretrieve(raw, '/content/colab_bootstrap.py')
    _BOOTSTRAP_SCRIPT = Path('/content/colab_bootstrap.py')
exec(_BOOTSTRAP_SCRIPT.read_text(encoding='utf-8'), globals())


In [ ]:
# Locked-test preflight.
baseline_run = WORKSPACE / 'runs' / 'v3_forecast_baselines'
ptcst_run = WORKSPACE / 'runs' / 'v3_ptcst_seed_sweep' / 'seed_7'
ablation_run = WORKSPACE / 'runs' / 'v3_ptcst_ablations'
for required in [baseline_run / 'forecasts.parquet', ptcst_run / 'metrics.json', ptcst_run / 'protocol_lock.json', ablation_run / 'portfolio_metrics_summary.parquet']:
    print(required, 'exists=', required.exists())
    if not required.exists(): raise FileNotFoundError(f'Missing prerequisite: {required}. Complete earlier notebooks first.')
print('Protocol lock present; proceeding to locked evaluation.')


In [ ]:
# Expanding quarterly walk-forward.
import subprocess, sys
walk_run = WORKSPACE / 'runs' / 'v3_walk_forward'
cmd = [sys.executable, str(REPO / 'scripts' / 'run_v3_walk_forward.py'), '--data-root', str(DATA_ROOT), '--run-dir', str(walk_run)]
result = subprocess.run(cmd, check=False, capture_output=True, text=True)
if result.stdout: print(result.stdout)
if result.stderr: print('STDERR:\n' + result.stderr)
if result.returncode != 0: raise RuntimeError(f'walk-forward failed: {result.stderr or result.stdout}')
print('Walk-forward completed:', walk_run)


In [ ]:
# Date-level forecast statistical tests.
stats_run = WORKSPACE / 'runs' / 'v3_statistical_tests'
cmd = [sys.executable, str(REPO / 'scripts' / 'run_v3_statistical_tests.py'), '--forecast-runs', f'baseline={baseline_run}', f'ptcst={ptcst_run}', '--run-dir', str(stats_run)]
result = subprocess.run(cmd, check=False, capture_output=True, text=True)
if result.stdout: print(result.stdout)
if result.stderr: print('STDERR:\n' + result.stderr)
if result.returncode != 0: raise RuntimeError(f'statistical tests failed: {result.stderr or result.stdout}')
import json
print(json.dumps(json.loads((stats_run / 'statistical_tests.json').read_text()), indent=2))


In [ ]:
# Persist M8–M9 outputs to Drive.
from shutil import copytree
drive_runs = Path('/content/drive/MyDrive/kltn/runs')
drive_runs.mkdir(parents=True, exist_ok=True)
for name, source in [('v3_walk_forward', walk_run), ('v3_statistical_tests', stats_run)]:
    copytree(source, drive_runs / name, dirs_exist_ok=True)
print('M8–M9 artifacts synced to:', drive_runs)
